# HARVEY + CATA on Kaggle (free GPU)

This notebook contains the modified source code. No code-dataset upload or GitHub clone is needed.

1. Upload this `.ipynb` into a new Kaggle notebook.
2. In **Settings / Session options**, select an available **GPU** accelerator and enable **Internet** (needed for CIFAR-10 and any missing packages).
3. Run the cells in order. Start with `PROFILE = "quick"` below.
4. Download `harvey-cata-results.zip` after completion. Use **Save Version → Save & Run All** for a saved notebook run and its output files.

Kaggle provides a limited weekly GPU allowance; check your own remaining quota and the session time limit in its UI. This notebook uses **one GPU**, including when two are assigned. No paid service or credentials are used. If GPU options are disabled, check account verification, remaining quota, and availability in Kaggle.

**Method:** CATA replaces HARVEY's benign-set unlearning in its poison-reference model. The original HARVEY poison-learning, RCE splitting, meta-splitting, and final clean-model training remain. This is a classifier hybrid; its final application base differs from original CATA Eq. 16, as explained in the included `README_CATA.md`.

In [ ]:
# Start here. Change PROFILE to "full" only after a successful quick run.
PROFILE = "full"     # "quick" or "full"
ATTACK = "badnets"    # "badnets" or "blend"
UNLEARNING = "GA"   # "CATA", or "GA" for original HARVEY unlearning
SEED = 1
BATCH_SIZE = 128      # reduce to 64 if an out-of-memory error occurs

assert PROFILE in {"quick", "full"}
assert ATTACK in {"badnets", "blend"}
assert UNLEARNING in {"CATA", "GA"}
assert BATCH_SIZE >= 2

## 1. Check the GPU and dependencies

Keep Kaggle's preinstalled PyTorch and torchvision versions. Only missing helper packages are installed. If this cell reports no CUDA GPU, select a GPU in notebook settings and restart the session.

In [ ]:
import os
import sys
import subprocess
import importlib.util
from pathlib import Path

os.environ["MPLCONFIGDIR"] = "/kaggle/working/.cache/matplotlib"
os.environ["XDG_CACHE_HOME"] = "/kaggle/working/.cache"
os.environ["MPLBACKEND"] = "Agg"
import torch
import torchvision

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__, "torchvision:", torchvision.__version__)
if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU. Enable a GPU accelerator in Kaggle settings, then restart the session.")
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {props.name}, {props.total_memory / 2**30:.1f} GiB")
print("Training uses GPU 0 only.")

helpers = {"yaml": "PyYAML", "sklearn": "scikit-learn", "psutil": "psutil",
           "PIL": "Pillow", "tqdm": "tqdm", "matplotlib": "matplotlib"}
missing = [package for module, package in helpers.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *missing], check=True)
print("Dependencies ready.")

## 2. Unpack the bundled implementation

Everything is written under `/kaggle/working`, which is writable. The embedded payload is the Python source, configs, tests, documentation, and trigger images; it is not a model or dataset.

In [ ]:
import base64
import io
import zipfile

CODE_DIR = Path("/kaggle/working/harvey-cata")
CODE_DIR.mkdir(parents=True, exist_ok=True)
SOURCE_ARCHIVE = (
    '...'
)
with zipfile.ZipFile(io.BytesIO(base64.b64decode(SOURCE_ARCHIVE))) as archive:
    for member in archive.infolist():
        destination = (CODE_DIR / member.filename).resolve()
        if CODE_DIR.resolve() not in destination.parents:
            raise ValueError("Unexpected path in source bundle")
    archive.extractall(CODE_DIR)
print("Source ready:", CODE_DIR)
print((CODE_DIR / "README_CATA.md").read_text()[:750])

## 3. Validate the unlearning implementation

These ten tests run on CPU and verify the equations and integration. They do not measure backdoor removal quality.

In [ ]:
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"],
               cwd=CODE_DIR, check=True)

## 4. Configure the run

Both profiles use all 50,000 CIFAR-10 training images and the standard test split, with 10% training poisoning.

| Setting | Quick trial | Full experiment |
|---|---:|---:|
| Initialization epochs | 2 | 20 |
| Maximum reference rounds | 3 | 20 |
| Poison-learning epochs per round | 2 | 14 |
| CATA adaptation epochs per round | 1 | 4 |
| Final clean-model epochs | 5 | 100 |

The quick profile checks execution and gives a timing baseline; its metrics are **not converged research results**. The full profile preserves the supplied experiment's training schedule and CATA settings. HARVEY may stop reference rounds early when its benign-set size stabilizes.

There is no validated wall-clock estimate for Kaggle. A full run may exceed the free session limit. Use the quick run's timing to decide whether to start it. This implementation does not support resuming CATA history after interruption; download outputs and logs after each completed run.

In [ ]:
import json
import yaml
from datetime import datetime, timezone

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
RUN_DIR = Path("/kaggle/working/harvey-cata-runs") / f"{PROFILE}-{ATTACK}-{UNLEARNING}-seed{SEED}-{stamp}"
RUN_DIR.mkdir(parents=True, exist_ok=False)
base_config = CODE_DIR / "conf" / f"ResNet18_cifar10_SL_{ATTACK}_rate010.yml"
config = yaml.safe_load(base_config.read_text())
config["DATA_ROOT"] = "/kaggle/working/datasets"
config["results_folder"] = str(RUN_DIR / "results")
config["BATCH"] = BATCH_SIZE
config["UNLEARNING"] = UNLEARNING
config["plot_hist"] = False
config["plot_tsne"] = False
config["print_cls_dist"] = False
if PROFILE == "quick":
    config["CYCLES"] = 3
    config["BACKDOOR"].update(warmup_epochs=2, ref_epochs=2, ref_c1epochs=2)
    config["CATA"]["EPOCHS"] = 1
    config["OPTIMIZER_BACKBONE"].update(EPOCH_TL=5, MILESTONES=[3, 4])
CONFIG_PATH = RUN_DIR / "config.yml"
CONFIG_PATH.write_text(yaml.safe_dump(config, sort_keys=False))

# Upstream caches poison indices in the source directory. Regenerate them for
# this seed; the train/unlabeled dataset views still share the same indices.
for cached in (CODE_DIR / "trigger").glob("*_poison_ids.npy"):
    cached.unlink()

metadata = {"profile": PROFILE, "attack": ATTACK, "unlearning": UNLEARNING,
            "seed": SEED, "batch_size": BATCH_SIZE,
            "gpu": torch.cuda.get_device_name(0), "torch": torch.__version__,
            "source_upstream_commit": "6641f5fedef0f654ab019aaebf6f3de24901bd9b",
            "started_utc": stamp, "status": "configured"}
(RUN_DIR / "run_info.json").write_text(json.dumps(metadata, indent=2))
print("Run folder:", RUN_DIR)
print(CONFIG_PATH.read_text())

## 5. Train

The first run downloads CIFAR-10 from torchvision's dataset source. Keep Internet enabled. Output streams below and is saved in `train.log`. Training failure or interruption triggers result packaging before the error is raised. Files include the actual configuration and environment details.

Reference-model clean accuracy is expected to decrease: that model is intended to specialize in poisoned samples. Judge utility using the **final clean model**, trained in Stage 4. Lower final attack success rate is better.

In [ ]:
import re
import time
import shutil

RESULT_ZIP = Path("/kaggle/working/harvey-cata-results.zip")
def package_results():
    # Keep the source used for the run with the results for reproducibility.
    source_copy = RUN_DIR / "source.zip"
    source_copy.write_bytes(base64.b64decode(SOURCE_ARCHIVE))
    shutil.make_archive(str(RESULT_ZIP.with_suffix("")), "zip", root_dir=RUN_DIR)
    print("Packaged:", RESULT_ZIP)

command = [sys.executable, "-u", "train.py", "--config_file", str(CONFIG_PATH),
           "--seed", str(SEED), "--cuda_visible_device", "0"]
process = None
started = time.monotonic()
metadata["status"] = "running"
(RUN_DIR / "run_info.json").write_text(json.dumps(metadata, indent=2))
try:
    with (RUN_DIR / "train.log").open("w", buffering=1) as log:
        process = subprocess.Popen(command, cwd=CODE_DIR, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line, end="")
            log.write(line)
        returncode = process.wait()
    metadata["returncode"] = returncode
    metadata["status"] = "completed" if returncode == 0 else "failed"
    if returncode:
        raise RuntimeError(f"Training exited with code {returncode}. See train.log in the result ZIP.")
    # Only a successful complete run is summarized as a final-model result.
    log_text = (RUN_DIR / "train.log").read_text()
    stage4 = log_text.split("Stage-4: Final clean model training")[-1]
    matches = re.findall(r"test_acc\(bng/bad\):\s*([\d.]+)/([\d.]+)", stage4)
    if "Stage-4: Final clean model training" in log_text and matches:
        clean, asr = map(float, matches[-1])
        metadata["final_clean_accuracy_percent"] = clean
        metadata["final_attack_success_rate_percent"] = asr
        print(f"\nFinal clean accuracy: {clean:.2f}% | Attack success rate: {asr:.2f}%")
    if PROFILE == "quick":
        print("Quick-profile result: execution check only; training is intentionally shortened.")
except BaseException:
    if process is not None and process.poll() is None:
        process.terminate()
        try:
            process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
    if metadata["status"] == "running":
        metadata["status"] = "interrupted_or_failed"
    raise
finally:
    metadata["elapsed_minutes"] = round((time.monotonic() - started) / 60, 2)
    (RUN_DIR / "run_info.json").write_text(json.dumps(metadata, indent=2))
    package_results()
    print("Elapsed minutes:", metadata["elapsed_minutes"])

## 6. Download and save

Run this cell even if training failed to get the logs. The ZIP contains:

- The final model (`*_latest_target_model.pt`) and split indices when training completed.
- Training log, selected configuration, environment/timing metadata, and final metrics when available.
- Source code used by the run.

These are final-output files, **not a resumable CATA checkpoint**. Session termination by Kaggle can bypass cleanup; intermediate output availability then depends on Kaggle's saved version/session behavior.

Download the ZIP, then save a notebook version with outputs and stop the GPU session when done. Re-running with a different seed/attack creates a new run folder; the download ZIP contains the most recent run.

In [ ]:
from IPython.display import FileLink, display
if RESULT_ZIP.exists():
    display(FileLink(str(RESULT_ZIP), result_html_prefix="Download results: "))
else:
    print("Run the training cell first.")
print(json.dumps(metadata, indent=2))

## Troubleshooting

- **No GPU / accelerator disabled:** check Kaggle account verification, remaining quota, and available accelerators. The full pipeline requires CUDA.
- **Download error:** enable Internet and retry the dataset download. The notebook expects torchvision's CIFAR-10 file layout; a random CSV-format CIFAR dataset is not interchangeable.
- **GPU out of memory:** restart the session, set `BATCH_SIZE = 64`, and rerun. This changes optimization batch size, so record it when comparing experiments.
- **Session limit / quota:** start with quick mode and inspect elapsed time. Full CATA resume across sessions is not implemented. Do not assume a full run fits a free session.
- **Poor quick-run accuracy:** expected after only five final training epochs; use full settings and matched seeds for research comparisons.

References: [Kaggle notebooks](https://www.kaggle.com/docs/notebooks), [Kaggle GPU usage](https://www.kaggle.com/docs/efficient-gpu-usage), [HARVEY source](https://github.com/intellisec/harvey). See bundled `README_CATA.md` for method details and citations.